#### Right Now we are mainly focusing on segregating the categories into Fiction or Nonfiction only, in future we  will try to separate it into smaller and more niche categories but at the same time we also need to make sure that this doesn't lead to some unwanted categories

In [25]:
import pandas as pd
from fontTools.varLib.avar.plan import makeDesignspaceSnippet
from jinja2.utils import missing
from matplotlib import category
from sympy.multipledispatch.dispatcher import ambiguity_register_error_ignore_dup

books = pd.read_csv('cleaned_data.csv')

In [26]:
books["categories"].value_counts().reset_index()

,categories,count
0,Fiction,2111
1,Juvenile Fiction,390
2,Biography & Autobiography,311
3,History,207
4,Literary Criticism,124
...,...,...
474,Human-animal relationships,1
475,Imperialism,1
476,Aged women,1
477,Humorous stories,1


In [27]:
category_mapping = {
'Fiction': "Fiction",
'Juvenile Fiction': "Children's Fiction",
'Biography & Autobiography': "Nonfiction",
'History': "Nonfiction",
'Literary Criticism': "Nonfiction",
'Philosophy': "Nonfiction",
'Religion': "Nonfiction",
'Comics & Graphic Novels': "Fiction",
'Drama': "Fiction",
'Juvenile Nonfiction': "Children's Nonfiction",
'Science': "Nonfiction",
'Poetry': "Fiction"
}

books["Simple_Categories"] = books["categories"].map(category_mapping)

In [28]:
books[~books["Simple_Categories"].isna()]
# Now we for simplification purposes we are (for now) just going to be using 2 categories here
# In future we shall group all the categores to the top 10 categories of our dataset

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,words_with_subtitle,Simple_Categories
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,9780002005883 A NOVEL THAT READERS and critics...,2004.0,3.85,247.0,361.0,Gilead,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"9780006178736 A memorable, mesmerizing heroine...",1993.0,3.93,512.0,29532.0,Rage of angels,Fiction
8,9780006482079,0006482074,Warhost of Vastmark,Janny Wurts,Fiction,http://books.google.com/books/content?id=uOL0f...,9780006482079 Tricked once more by his wily ha...,1995.0,4.03,522.0,2966.0,Warhost of Vastmark,Fiction
30,9780006646006,000664600X,Ocean Star Express,Mark Haddon;Peter Sutton,Juvenile Fiction,http://books.google.com/books/content?id=I2QZA...,9780006646006 Joe and his parents are enjoying...,2002.0,3.50,32.0,1.0,Ocean Star Express,Children's Fiction
46,9780007121014,0007121016,Taken at the Flood,Agatha Christie,Fiction,http://books.google.com/books/content?id=3gWlx...,9780007121014 A Few Weeks After Marrying An At...,2002.0,3.71,352.0,8852.0,Taken at the Flood,Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5178,9781933648279,1933648279,Night Has a Thousand Eyes,Cornell Woolrich,Fiction,http://books.google.com/books/content?id=3Gk6s...,"9781933648279 ""Cornell Woolrich's novels defin...",2007.0,3.77,344.0,680.0,Night Has a Thousand Eyes,Fiction
5188,9784770028969,4770028962,Coin Locker Babies,村上龍,Fiction,http://books.google.com/books/content?id=87DJw...,9784770028969 Rescued from the lockers in whic...,2002.0,3.75,393.0,5560.0,Coin Locker Babies,Fiction
5189,9788122200850,8122200850,"Cry, the Peacock",Anita Desai,Fiction,http://books.google.com/books/content?id=_QKwV...,9788122200850 This book is the story of a youn...,1980.0,3.22,218.0,134.0,"Cry, the Peacock",Fiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,9788185300535 This collection of the timeless ...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,Nonfiction


In [29]:
from dotenv import load_dotenv

load_dotenv()

False

In [30]:
from transformers import pipeline
classifier = pipeline("zero-shot-classification",
                      model="facebook/bart-large-mnli",
                      device="cuda")

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [31]:
# from tqdm import tqdm
#
# actual_cats = []
# predicted_cats = []
# fiction_categories = ["Fiction", "Nonfiction"]
#
# for i in tqdm(range(0,300)):
#     sequence = books.loc[books["Simple_Categories"] == "Fiction", "description"].reset_index(drop=True)[i]
#     predicted_cats += generate_predictions(sequence, fiction_categories)
#     actual_cats += ["Fiction"]

#### In above we do sequential calls which is not optimal for gpu, because it can parallelise itself then

In [32]:
import numpy as np
from tqdm import tqdm

fiction_categories = ["Fiction", "Nonfiction"]
sequences = (
    books.loc[books["Simple_Categories"] == "Fiction", "description"]
    .reset_index(drop=True)
    .iloc[:300]
    .tolist()
)

actual_cats = ["Fiction"] * len(sequences)
predicted_cats = []

batch_size = 16  # raise to 32/64 if GPU memory allows, lower if you hit OOM

for output in tqdm(
    classifier(sequences, fiction_categories, batch_size=batch_size),
    total=len(sequences),
):
    max_score_idx = np.argmax(output["scores"])
    predicted_cats.append(output["labels"][max_score_idx])

100%|██████████| 300/300 [00:00<00:00, 93525.43it/s]


In [33]:
non_fic_sequences = (
    books.loc[books["Simple_Categories"] == "Nonfiction", "description"]
    .reset_index(drop=True)
    .iloc[:300]
    .tolist()
)

actual_cats += ["Nonfiction"] * len(non_fic_sequences)

for output in tqdm(
    classifier(non_fic_sequences, fiction_categories, batch_size=batch_size),
    total=len(non_fic_sequences),
):
    max_score_idx = np.argmax(output["scores"])
    predicted_cats.append(output["labels"][max_score_idx])

100%|██████████| 300/300 [00:00<00:00, 93345.04it/s]


In [35]:
preds_df = pd.DataFrame({"Actual_categories": actual_cats, "Predicted_categories": predicted_cats})
preds_df["correct_preds"] = np.where(preds_df["Actual_categories"] == preds_df["Predicted_categories"], 1, 0)
print(f"Accuracy: {preds_df["correct_preds"].sum()/len(preds_df)}")

Accuracy: 0.795


#### For a zero shot classifier the model isn't that bad, 80% Accuracy is workable for us(for now)

In [36]:
# For the missing ones we will pred using the zero shot model as we saw the accuracy is okaish
seq = []
isbns = []
missing_cats = books.loc[books["Simple_Categories"].isna(), ["isbn13","description"]].reset_index(drop=True)
for i in range(0, len(missing_cats)):
    seq += [missing_cats["description"][i]]
    isbns += [missing_cats["isbn13"][i]]

isbns = missing_cats["isbn13"].tolist()
seq = missing_cats["description"].tolist()

pred_cats = []
for output in tqdm(classifier(seq, fiction_categories, batch_size=batch_size), total=len(seq)):
    pred_cats.append(output["labels"][np.argmax(output["scores"])])

100%|██████████| 1454/1454 [00:00<00:00, 106684.59it/s]


In [37]:
miss_pred_df = pd.DataFrame({"isbn13" : isbns, "pred_cats" : pred_cats})

In [38]:
books = pd.merge(books, miss_pred_df, on="isbn13", how="left")
books["Simple_Categories"] = np.where(books["Simple_Categories"].isna(), books["pred_cats"], books["Simple_Categories"])

In [39]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,words_with_subtitle,Simple_Categories,pred_cats
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,9780002005883 A NOVEL THAT READERS and critics...,2004.0,3.85,247.0,361.0,Gilead,Fiction,NaN
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,9780002261982 A new 'Christie for Christmas' -...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,Fiction,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"9780006178736 A memorable, mesmerizing heroine...",1993.0,3.93,512.0,29532.0,Rage of angels,Fiction,NaN
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,9780006280897 Lewis' work on the nature of lov...,2002.0,4.15,170.0,33684.0,The Four Loves,Nonfiction,Nonfiction
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"9780006280934 ""In The Problem of Pain, C.S. Le...",2002.0,4.09,176.0,37569.0,The Problem of Pain,Nonfiction,Nonfiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,9788172235222 On A Train Journey Home To North...,2003.0,2.93,324.0,0.0,Mistaken Identity,Nonfiction,Nonfiction
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,9788173031014 This book tells the tale of a ma...,2002.0,3.70,175.0,24.0,Journey to the East,Nonfiction,Nonfiction
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,9788179921623 Wisdom to Create a Life of Passi...,2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,Fiction,Fiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,9788185300535 This collection of the timeless ...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,Nonfiction,NaN


In [47]:
final_books = books.drop(columns=["pred_cats"])
final_books.head()

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,words_with_subtitle,Simple_Categories
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,9780002005883 A NOVEL THAT READERS and critics...,2004.0,3.85,247.0,361.0,Gilead,Fiction
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,9780002261982 A new 'Christie for Christmas' -...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"9780006178736 A memorable, mesmerizing heroine...",1993.0,3.93,512.0,29532.0,Rage of angels,Fiction
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,9780006280897 Lewis' work on the nature of lov...,2002.0,4.15,170.0,33684.0,The Four Loves,Nonfiction
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"9780006280934 ""In The Problem of Pain, C.S. Le...",2002.0,4.09,176.0,37569.0,The Problem of Pain,Nonfiction
